In [6]:
import pandas as pd
import numpy as np

In [2]:
df_green = pd.read_parquet('green_tripdata_2025-11.parquet')
df_green.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,1.0,0.74,7.2,...,0.5,1.94,0.0,NaN,1.0,11.64,1.0,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,2.0,0.95,7.2,...,0.5,0.00,0.0,NaN,1.0,9.70,2.0,1.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,1.0,2.19,13.5,...,0.5,5.00,0.0,NaN,1.0,21.00,1.0,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,1.0,5.44,24.7,...,0.5,0.50,0.0,NaN,1.0,27.70,1.0,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,1.0,3.20,18.4,...,1.5,1.00,0.0,NaN,1.0,24.65,1.0,1.0,2.75,0.0


In [3]:
df_zones = pd.read_csv('taxi_zone_lookup.csv')
df_zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [4]:
q3 = df_green[df_green['lpep_pickup_datetime'].between('2025-11-01', '2025-12-01', inclusive='left')]

In [15]:
sum(np.where(q3['trip_distance'] <= 1, 1, 0))

np.int64(8007)

In [19]:
q4 = df_green[df_green['trip_distance'] < 100].sort_values(by='trip_distance', ascending=False)
q4.head(1)['lpep_pickup_datetime']

18867   2025-11-14 15:36:27
Name: lpep_pickup_datetime, dtype: datetime64[us]

In [21]:
df_green.head().columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'cbd_congestion_fee'],
      dtype='str')

In [24]:
df_zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [33]:
q5 = pd.merge(df_green[['PULocationID', 'total_amount']], df_zones[['LocationID', 'Zone']], how='left', left_on='PULocationID', right_on='LocationID')
q5.drop(columns=['PULocationID', 'LocationID'], inplace=True)
q5

,total_amount,Zone
0,11.64,East Harlem North
1,9.70,East Harlem North
2,21.00,Elmhurst/Maspeth
3,27.70,Morningside Heights
4,24.65,Morningside Heights
...,...,...
46907,34.72,Crotona Park
46908,16.13,East Harlem North
46909,44.42,Brooklyn Heights
46910,26.17,Bayside


In [35]:
q5.groupby('Zone').agg(sum).sort_values('total_amount', ascending=False)

,total_amount
Zone,
East Harlem North,257684.70
East Harlem South,126791.81
Morningside Heights,49146.64
Jamaica,46490.74
Central Park,45626.50
...,...
Mariners Harbor,27.23
Battery Park City,21.04
Roosevelt Island,16.50


In [55]:
EHN_ID = df_zones.loc[df_zones['Zone'] == 'East Harlem North', 'LocationID'].values[0]
q6 = df_green.loc[(df_green['PULocationID'] == EHN_ID) & (df_green['lpep_pickup_datetime'].dt.to_period('M') == '2025-11'), ][['DOLocationID', 'tip_amount']].groupby('DOLocationID', as_index=False).agg(max).sort_values(by='tip_amount', ascending=False)
q6.head()

,DOLocationID,tip_amount
134,263,81.89
61,138,50.00
29,74,45.00
68,146,34.25
136,265,28.90


In [74]:
df_zones.loc[df_zones['LocationID'] == q6.iloc[0, 0]]['Zone']

262    Yorkville West
Name: Zone, dtype: str